# Notebook 02 — Feature Extraction
### WID2003 Cognitive Science | FSKTM, Universiti Malaya

---

## Overview

Raw gaze coordinates change every millisecond and cannot be fed directly into a classifier. This notebook transforms the cleaned gaze data into a set of **meaningful, stable eye-tracking features** per participant per task — the feature matrix (`X`) that the models will learn from.

**Inputs**
| File | Description |
|---|---|
| `data/processed/metrics_clean.parquet` | Cleaned Metrics data (from Notebook 01) |
| `data/processed/raw_gaze_clean.parquet` | Cleaned raw gaze stream (from Notebook 01) |
| `data/external/task_correct_aoi_map.json` | Maps each task to its correct AOI and distractor AOIs |

**Output**
| File | Description |
|---|---|
| `data/processed/features_per_task.parquet` | One row per (participant × task), ~35 feature columns |

---

## Learning Objectives

By the end of this notebook, you should be able to:

1. Explain what an **eye-tracking feature** is and why raw gaze coordinates are not suitable as direct model inputs
2. Distinguish between **correct-AOI features** and **distractor features**, and explain the cognitive significance of each
3. Compute derived features including **scanpath length**, **re-fixation count**, and **AOI dwell ratio** from a raw gaze sequence
4. Explain how **pupil diameter change** can serve as a proxy for cognitive load over time

---

## Background

### What Is a Feature?

A **feature** is a measurable property that summarises the behaviour of one participant across a trial. Instead of thousands of gaze samples, we extract one number per participant per metric — a stable statistic that reflects how they searched.

### AOI-Based Feature Split

For each task, features are extracted separately for:
- The **correct AOI** (e.g. `C1` for `Crown`) — the correct target region
- The **distractor AOIs** (e.g. `C2`–`C10`) — the wrong regions

This split lets us capture how much time a participant wasted on distractors compared to the target.

### Features Extracted

**From the Metrics file** (pre-computed by Tobii Pro Lab per AOI):

| Feature | Cognitive interpretation |
|---|---|
| `correct_Total_duration_of_fixations` | How long did the participant look at the target? |
| `correct_Number_of_fixations` | How many times did their gaze land on the target? |
| `correct_Time_to_first_fixation` | How quickly did they detect the target? |
| `correct_Number_of_Visits` | How many times did they return to the target? |
| `distractor_sum_Total_duration_of_fixations` | Total time spent on wrong regions |
| `Average_pupil_diameter` | Mean arousal/load during the trial |

**From the raw gaze stream** (computed in this notebook):

| Feature | How it is computed |
|---|---|
| `correct_aoi_dwell_ratio` | Frames with `AOI hit [task - correct_aoi] == 1` ÷ total valid frames |
| `distractor_dwell_ratio` | Frames on any distractor AOI ÷ total valid frames |
| `scanpath_length` | Sum of pixel distances between consecutive fixation centroids |
| `refixation_count` | Number of `0 → 1` transitions in the correct AOI hit column |
| `pupil_dilation_change` | Mean pupil (2nd half of trial) − mean pupil (1st half) |

### The `correct_aoi_dwell_ratio`

This is the most important single feature in the study. It answers: *"Of all the time the participant looked at valid screen regions, what fraction was spent on the correct answer?"*

A **high-performing** student typically shows a high dwell ratio early — their gaze is drawn to the target quickly and stays there.

---

## Discussion Questions

1. Why do we use `correct_aoi_dwell_ratio` instead of `correct_Total_duration_of_fixations`? What does normalisation by total valid frames correct for?
2. A participant has a very low `correct_aoi_dwell_ratio` but still found the correct answer. What might explain this pattern?
3. What does a high `refixation_count` on the correct AOI suggest about the participant's cognitive process (e.g. uncertainty, verification strategy)?
4. Why might `pupil_dilation_change` be positive for harder tasks and negative or near-zero for easy tasks?
5. The `scanpath_length` is measured in pixels. What are the limitations of comparing scanpath lengths across participants who may have sat at different distances from the screen?

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/WID2003

!python -m pip install -q -r requirements.txt

# Important: notebook imports assume the working directory is notebooks/
%cd /content/drive/MyDrive/WID2003/notebooks

In [1]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
from scipy.spatial.distance import euclidean

from src.config import (
    METRICS_CLEAN_PKL, RAWGAZE_CLEAN_PKL, FEATURES_PER_TASK, AOI_MAP_JSON,
    MetricsCols, ExportCols, METRICS_FEATURE_COLS, TASKS
)

pd.set_option('display.max_columns', 50)

## 1. Load inputs

In [2]:
metrics = pd.read_parquet(METRICS_CLEAN_PKL)
gaze    = pd.read_parquet(RAWGAZE_CLEAN_PKL)

with open(AOI_MAP_JSON) as f:
    aoi_map = json.load(f)

# Strip _comment key
aoi_map = {k: v for k, v in aoi_map.items() if not k.startswith('_')}

print("AOI map loaded for tasks:", list(aoi_map.keys()))
print("Metrics shape:", metrics.shape)
print("Gaze shape:",    gaze.shape)

AOI map loaded for tasks: ['Crown', 'findDice', 'Hat', 'Iguana', 'Rabbit', 'Shoe', 'T1_Prisoner-15sec', 'T2_Ring-15sec', 'T3_Umbrella-15 sec', 'T4_Pen-15sec', 'T5_Fish-15sec', 'T6_Heart-15sec', 'Toothbrush']
Metrics shape: (39000, 53)
Gaze shape: (4538514, 154)


## 2. Features from Metrics file

For each (Participant, Media/task), extract separate feature rows for:
- correct AOI
- all distractors (aggregated)
- total (all AOIs combined)

In [3]:
def extract_metrics_features(metrics_df, aoi_map):
    """Return a DataFrame with one row per (participant, task)."""
    records = []

    for task in TASKS:
        if task not in aoi_map:
            print(f"WARNING: {task} not in aoi_map, skipping")
            continue

        correct_aoi    = aoi_map[task]['correct_aoi']
        distractor_aois = aoi_map[task]['distractor_aois']

        task_df = metrics_df[metrics_df[MetricsCols.MEDIA].str.contains(task, na=False, case=False)]

        for participant, grp in task_df.groupby(MetricsCols.PARTICIPANT):
            row = {'participant_id': participant, 'task': task}

            # --- Correct AOI metrics ---
            correct_rows = grp[grp[MetricsCols.AOI].str.strip() == correct_aoi]
            if len(correct_rows) > 0:
                cr = correct_rows.iloc[0]
                for col in METRICS_FEATURE_COLS:
                    row[f'correct_{col}'] = cr.get(col, np.nan)
            else:
                for col in METRICS_FEATURE_COLS:
                    row[f'correct_{col}'] = np.nan

            # --- Distractor metrics (mean across distractor AOIs) ---
            dist_rows = grp[grp[MetricsCols.AOI].str.strip().isin(distractor_aois)]
            if len(dist_rows) > 0:
                for col in [MetricsCols.TOTAL_FIX_DUR, MetricsCols.NUM_FIXATIONS,
                            MetricsCols.TOTAL_VISIT_DUR, MetricsCols.NUM_VISITS]:
                    row[f'distractor_sum_{col}'] = dist_rows[col].sum()
                    row[f'distractor_mean_{col}'] = dist_rows[col].mean()
            else:
                for col in [MetricsCols.TOTAL_FIX_DUR, MetricsCols.NUM_FIXATIONS,
                            MetricsCols.TOTAL_VISIT_DUR, MetricsCols.NUM_VISITS]:
                    row[f'distractor_sum_{col}']  = np.nan
                    row[f'distractor_mean_{col}'] = np.nan

            # --- AOI transition count (proxy: total visits across all AOIs) ---
            row['total_aoi_visits'] = grp[MetricsCols.NUM_VISITS].sum()

            records.append(row)

    return pd.DataFrame(records)


metrics_features = extract_metrics_features(metrics, aoi_map)
print(f"Metrics features shape: {metrics_features.shape}")
metrics_features.head()

Metrics features shape: (1690, 30)


,participant_id,task,correct_Total_duration_of_fixations,correct_Average_duration_of_fixations,correct_Number_of_fixations,correct_Time_to_first_fixation,correct_Duration_of_first_fixation,correct_Total_duration_of_Visit,correct_Number_of_Visits,correct_Time_to_first_Visit,correct_Average_duration_of_Visit,correct_Total_duration_of_Glances,correct_Number_of_Glances,correct_Number_of_saccades_in_AOI,correct_Peak_velocity_of_entry_saccade,correct_Peak_velocity_of_exit_saccade,correct_Average_pupil_diameter,correct_Average_eye_openness,correct_Time_to_first_mouse_click,correct_Time_from_first_fixation_to_mouse_click,correct_Number_of_mouse_clicks,distractor_sum_Total_duration_of_fixations,distractor_mean_Total_duration_of_fixations,distractor_sum_Number_of_fixations,distractor_mean_Number_of_fixations,distractor_sum_Total_duration_of_Visit,distractor_mean_Total_duration_of_Visit,distractor_sum_Number_of_Visits,distractor_mean_Number_of_Visits,total_aoi_visits
0,vt1,Crown,1400,467.0,3,4330.0,175.0,1400,3,4330.0,467.0,1500,3,0,219.71,334.93,3.36972,NaN,16170.0,11839.0,1,14700,816.666667,46,2.555556,14866,825.888889,38,2.111111,250
1,vt10,Crown,1615,808.0,2,2844.0,300.0,1615,2,2844.0,808.0,1674,2,0,180.31,220.94,4.25706,NaN,4610.0,1766.0,1,1700,94.444444,10,0.555556,1718,95.444444,8,0.444444,140
2,vt100,Crown,776,388.0,2,1842.0,125.0,792,1,1842.0,792.0,834,1,1,385.11,NaN,3.26728,NaN,NaN,NaN,0,1568,87.111111,10,0.555556,1618,89.888889,8,0.444444,171
3,vt101,Crown,1654,551.0,3,4181.0,250.0,1662,2,4181.0,831.0,1720,2,1,190.28,223.24,3.59316,NaN,6346.0,2166.0,1,3444,191.333333,16,0.888889,3794,210.777778,14,0.777778,162
4,vt102,Crown,0,NaN,0,NaN,NaN,0,0,NaN,NaN,0,0,0,NaN,NaN,NaN,NaN,5272.0,NaN,1,2532,140.666667,14,0.777778,2598,144.333333,10,0.555556,155


## 3. Features from raw gaze stream

Computed per (participant, task):
- `correct_aoi_dwell_ratio` — fraction of total valid frames on correct AOI
- `distractor_dwell_ratio` — fraction on any distractor AOI
- `scanpath_length` — cumulative Euclidean distance between consecutive fixation points
- `refixation_count` — number of returns to correct AOI after leaving
- `pupil_dilation_change` — pupil diameter second-half minus first-half

In [4]:
def build_aoi_hit_col(task, aoi):
    """Build the column name pattern used in the Data export: 'AOI hit [task - aoi]'."""
    return f"AOI hit [{task} - {aoi}]"


def compute_scanpath_length(fix_x, fix_y):
    """Cumulative Euclidean distance between consecutive fixation centroids."""
    coords = list(zip(fix_x, fix_y))
    if len(coords) < 2:
        return 0.0
    return sum(euclidean(coords[i], coords[i+1]) for i in range(len(coords)-1))


def count_refixations(aoi_hit_series):
    """Count how many times gaze returns to AOI after leaving (0->1 transitions)."""
    transitions = aoi_hit_series.diff().fillna(0)
    return int((transitions == 1).sum())


def extract_gaze_features(gaze_df, aoi_map):
    records = []

    for task in TASKS:
        if task not in aoi_map:
            continue

        correct_aoi     = aoi_map[task]['correct_aoi']
        distractor_aois = aoi_map[task]['distractor_aois']
        correct_col     = build_aoi_hit_col(task, correct_aoi)
        distractor_cols = [build_aoi_hit_col(task, a) for a in distractor_aois
                           if build_aoi_hit_col(task, a) in gaze_df.columns]

        if correct_col not in gaze_df.columns:
            print(f"WARNING: column '{correct_col}' not found — skipping {task}")
            continue

        task_df = gaze_df[gaze_df[ExportCols.PRESENTED_MEDIA].str.contains(task, na=False, case=False)]

        for participant, grp in task_df.groupby(ExportCols.PARTICIPANT_NAME):
            row = {'participant_id': participant, 'task': task}

            total_frames = len(grp)
            valid_frames = grp[ExportCols.GAZE_X].notna().sum()

            # Dwell ratios
            row['correct_aoi_dwell_ratio']    = grp[correct_col].sum() / max(valid_frames, 1)
            if distractor_cols:
                distractor_hits = grp[distractor_cols].any(axis=1).sum()
                row['distractor_dwell_ratio'] = distractor_hits / max(valid_frames, 1)
            else:
                row['distractor_dwell_ratio'] = np.nan

            # Scanpath length (fixation rows only)
            fix_rows = grp[grp[ExportCols.EYE_MOVEMENT_TYPE] == 'Fixation'].dropna(
                subset=[ExportCols.FIXATION_X, ExportCols.FIXATION_Y]
            )
            row['scanpath_length'] = compute_scanpath_length(
                fix_rows[ExportCols.FIXATION_X].values,
                fix_rows[ExportCols.FIXATION_Y].values
            )

            # Re-fixation count on correct AOI
            row['refixation_count'] = count_refixations(grp[correct_col])

            # Pupil dilation change (first half vs second half)
            pupil = grp[ExportCols.PUPIL_FILTERED].dropna()
            if len(pupil) >= 4:
                mid = len(pupil) // 2
                row['pupil_dilation_change'] = pupil.iloc[mid:].mean() - pupil.iloc[:mid].mean()
            else:
                row['pupil_dilation_change'] = np.nan

            row['valid_frames']  = int(valid_frames)
            row['total_frames']  = int(total_frames)

            records.append(row)

    return pd.DataFrame(records)


gaze_features = extract_gaze_features(gaze, aoi_map)
print(f"Gaze features shape: {gaze_features.shape}")
gaze_features.head()

Gaze features shape: (1690, 9)


,participant_id,task,correct_aoi_dwell_ratio,distractor_dwell_ratio,scanpath_length,refixation_count,pupil_dilation_change,valid_frames,total_frames
0,vt1,Crown,0.092939,0.460540,12722.654701,3,0.149962,1926,2045
1,vt10,Crown,0.451613,0.275986,2844.944658,2,0.379721,558,777
2,vt100,Crown,0.297468,0.297468,3323.817822,2,0.086348,316,318
3,vt101,Crown,0.326232,0.278296,6053.797981,3,0.194274,751,816
4,vt102,Crown,0.000000,0.243981,2320.263450,0,0.182706,623,673


## 4. Merge Metrics and Gaze features

In [5]:
if gaze_features.empty:
    print("WARNING: gaze_features is empty — Data Export TSV AOI columns not found.")
    print("         Re-export the Data Export TSV from Tobii Pro Lab with renamed AOIs,")
    print("         then re-run this notebook. Proceeding with metrics-only features.")
    features = metrics_features.copy()
else:
    features = pd.merge(
        metrics_features,
        gaze_features,
        on=['participant_id', 'task'],
        how='outer'
    )

print(f"Combined features shape: {features.shape}")
print(f"Participants: {features['participant_id'].nunique()}")
print(f"Tasks: {features['task'].unique()}")
features.head()

Combined features shape: (1690, 37)
Participants: 130
Tasks: ['Crown' 'Hat' 'Iguana' 'Rabbit' 'Shoe' 'T1_Prisoner-15sec'
 'T2_Ring-15sec' 'T3_Umbrella-15 sec' 'T4_Pen-15sec' 'T5_Fish-15sec'
 'T6_Heart-15sec' 'Toothbrush' 'findDice']


,participant_id,task,correct_Total_duration_of_fixations,correct_Average_duration_of_fixations,correct_Number_of_fixations,correct_Time_to_first_fixation,correct_Duration_of_first_fixation,correct_Total_duration_of_Visit,correct_Number_of_Visits,correct_Time_to_first_Visit,correct_Average_duration_of_Visit,correct_Total_duration_of_Glances,correct_Number_of_Glances,correct_Number_of_saccades_in_AOI,correct_Peak_velocity_of_entry_saccade,correct_Peak_velocity_of_exit_saccade,correct_Average_pupil_diameter,correct_Average_eye_openness,correct_Time_to_first_mouse_click,correct_Time_from_first_fixation_to_mouse_click,correct_Number_of_mouse_clicks,distractor_sum_Total_duration_of_fixations,distractor_mean_Total_duration_of_fixations,distractor_sum_Number_of_fixations,distractor_mean_Number_of_fixations,distractor_sum_Total_duration_of_Visit,distractor_mean_Total_duration_of_Visit,distractor_sum_Number_of_Visits,distractor_mean_Number_of_Visits,total_aoi_visits,correct_aoi_dwell_ratio,distractor_dwell_ratio,scanpath_length,refixation_count,pupil_dilation_change,valid_frames,total_frames
0,vt1,Crown,1400,467.0,3,4330.0,175.0,1400,3,4330.0,467.0,1500,3,0,219.71,334.93,3.36972,NaN,16170.0,11839.0,1,14700,816.666667,46,2.555556,14866,825.888889,38,2.111111,250,0.092939,0.460540,12722.654701,3,0.149962,1926,2045
1,vt1,Hat,5245,477.0,11,1684.0,333.0,5370,5,1684.0,1074.0,5520,5,6,251.08,291.63,3.33503,NaN,21287.0,19603.0,1,16486,915.888889,54,3.000000,16620,923.333333,48,2.666667,257,0.267787,0.387021,13433.040880,11,0.015173,2558,2618
2,vt1,Iguana,483,483.0,1,4112.0,483.0,483,1,4112.0,483.0,508,1,0,171.20,NaN,3.35847,NaN,NaN,NaN,0,19304,1072.444444,68,3.777778,19654,1091.888889,54,3.000000,256,0.016129,0.322581,22360.678215,1,0.012325,3596,3603
3,vt1,Rabbit,2515,838.0,3,3219.0,392.0,2540,2,3219.0,1270.0,2648,2,1,337.26,NaN,3.43547,NaN,6067.0,2848.0,1,2224,123.555556,12,0.666667,2256,125.333333,10,0.555556,235,0.522696,0.182944,6351.747480,3,0.140226,727,814
4,vt1,Shoe,2414,805.0,3,23140.0,283.0,2448,1,23140.0,2448.0,2489,1,2,204.84,NaN,3.41906,NaN,25540.0,2400.0,1,12986,721.444444,44,2.444444,13104,728.000000,38,2.111111,248,0.117858,0.254652,16947.928894,3,0.026431,3063,3178


## 5. Feature summary

In [6]:
print("Feature columns:")
feature_cols = [c for c in features.columns if c not in ['participant_id', 'task']]
print(f"  Total: {len(feature_cols)}")
for c in feature_cols:
    print(f"  {c}")

Feature columns:
  Total: 35
  correct_Total_duration_of_fixations
  correct_Average_duration_of_fixations
  correct_Number_of_fixations
  correct_Time_to_first_fixation
  correct_Duration_of_first_fixation
  correct_Total_duration_of_Visit
  correct_Number_of_Visits
  correct_Time_to_first_Visit
  correct_Average_duration_of_Visit
  correct_Total_duration_of_Glances
  correct_Number_of_Glances
  correct_Number_of_saccades_in_AOI
  correct_Peak_velocity_of_entry_saccade
  correct_Peak_velocity_of_exit_saccade
  correct_Average_pupil_diameter
  correct_Average_eye_openness
  correct_Time_to_first_mouse_click
  correct_Time_from_first_fixation_to_mouse_click
  correct_Number_of_mouse_clicks
  distractor_sum_Total_duration_of_fixations
  distractor_mean_Total_duration_of_fixations
  distractor_sum_Number_of_fixations
  distractor_mean_Number_of_fixations
  distractor_sum_Total_duration_of_Visit
  distractor_mean_Total_duration_of_Visit
  distractor_sum_Number_of_Visits
  distractor_mean_N

In [7]:
null_summary = features[feature_cols].isnull().mean().sort_values(ascending=False)
print("\nNull rate per feature:")
null_summary[null_summary > 0]


Null rate per feature:


correct_Average_eye_openness                       1.000000
correct_Time_from_first_fixation_to_mouse_click    0.518935
correct_Peak_velocity_of_exit_saccade              0.511243
correct_Time_to_first_mouse_click                  0.492899
correct_Peak_velocity_of_entry_saccade             0.259172
correct_Average_duration_of_fixations              0.198225
correct_Average_pupil_diameter                     0.198225
correct_Time_to_first_Visit                        0.198225
correct_Time_to_first_fixation                     0.198225
correct_Average_duration_of_Visit                  0.198225
correct_Duration_of_first_fixation                 0.198225
dtype: float64

## 6. Save

In [8]:
features.to_parquet(FEATURES_PER_TASK, index=False)
print(f"Saved: {FEATURES_PER_TASK}")
print(f"Shape: {features.shape}")

Saved: /home/wlsoo/WID2003/data/processed/features_per_task.parquet
Shape: (1690, 37)
